## Executive Summary

**Purpose:** Turn a target ratio, a number of planets and a module into a planetary gear set that meshes, assembles and fits: tooth counts, profile shifts, centre distance, speeds, and the space left between the planets and around the sun bore and ring.

**What it does:** `tooth_count_candidates` lists every sun/planet/ring combination near the target ratio that satisfies the coaxial, assembly and neighbour conditions. `PlanetarySet.build` picks profile shifts so the sun is not undercut and both meshes share one centre distance, and shortens the ring tips where they would dig into the planet roots. `module_limits` turns the envelope diameter and the sun shaft into the smallest and largest module that fit.

**Arrangement:** ring gear fixed, sun is the input, carrier is the output (the usual single-stage reducer). Ratio $i = n_s/n_c = 1 + z_r/z_s$.

## How It Fits Into the Gear Design Workflow

**Upstream:** `gears.involute` (`GearGeometry`, `mesh_check`, centre distance and shift relations).

**Downstream:** `gears.rating` (loads and stresses of the two meshes), `gears.components` (planet pin, bearing and carrier at the centre distance and planet bore found here), `gears.design` (the search over modules and tooth counts).

In [ ]:
#| default_exp gears.layout

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass, field

from emachines.gears.involute import (GearGeometry, MeshCheck, mesh_check, centre_distance,
                                      shift_sum_for_centre_distance)

## Tooth Counts: Theory

With sun $z_s$, planets $z_p$, ring $z_r$ and $N$ equally spaced planets:

1. **Ratio** (ring fixed): $i = 1 + z_r/z_s$.
2. **Coaxial condition.** Both meshes must have the same centre distance. With unshifted gears that needs $z_r = z_s + 2z_p$. Profile shift lets the ring differ by one tooth, $z_r = z_s + 2z_p + \delta$ with $\delta = \pm 1$, which opens up more ratios; the shifts then absorb the half-module mismatch.
3. **Assembly condition.** $N$ planets can be put in at equal angles only if $(z_s + z_r)/N$ is a whole number.
4. **Neighbour condition.** Adjacent planets must not touch: $2a_w\sin(\pi/N) > d_{a,p} + c$, where $c$ is the gap you need for oil and for the carrier posts that pass between the planets.
5. **Undercut.** A hobbed sun below about 17 teeth needs a positive shift; this module adds it automatically, so $z_s$ down to about 12 is allowed.
6. **Hunting teeth** (preferred, not required): if $z_s$ and $z_p$ share no common factor, and neither do $z_p$ and $z_r$, every tooth meets every mating tooth and wear evens out.

Speeds follow from the Willis equation $n_s - n_c = -(z_r/z_s)(n_r - n_c)$ with $n_r = 0$: the carrier turns at $n_s/i$, the planets spin on their pins at $n_{p/c} = -(n_s - n_c)\,z_s/z_p$.

In [ ]:
#| export
@dataclass(frozen=True)
class ToothCounts:
    """
    One sun/planet/ring combination.

    zs, zp, zr : int   Tooth counts
    n_planets  : int   Number of planets N
    ratio      : float i = 1 + zr/zs
    error      : float Relative error to the target ratio
    hunting    : bool  zs,zp and zp,zr have no common factor
    delta      : int   zr − (zs + 2 zp): 0 for an unshifted set, ±1 needs profile shift
    """
    zs: int
    zp: int
    zr: int
    n_planets: int
    ratio: float
    error: float
    hunting: bool
    delta: int


def tooth_count_candidates(ratio: float, n_planets: int, tol: float = 0.02,
                           zs_range: tuple[int, int] = (12, 40), zp_min: int = 12,
                           deltas: tuple[int, ...] = (0, -1, 1),
                           neighbour_gap: float = 1.0) -> list[ToothCounts]:
    """
    All tooth counts within `tol` of the target ratio that assemble with `n_planets` equally spaced
    planets and leave a gap of at least `neighbour_gap` modules between neighbouring planet tips.

    Sorted by ratio error in 0.5 % steps, then unshifted sets (δ = 0) first, hunting teeth first, smaller sun first.
    The neighbour check uses unshifted tip diameters; `PlanetarySet` repeats it with the final shifts.
    """
    if ratio <= 2:
        raise ValueError("a ring-fixed single stage gives i > 2")
    out = []
    for zs in range(zs_range[0], zs_range[1] + 1):
        zr_target = (ratio - 1) * zs
        for zr in range(int(zr_target * (1 - tol)) - 1, int(zr_target * (1 + tol)) + 2):
            i = 1 + zr / zs
            err = (i - ratio) / ratio
            if abs(err) > tol or (zs + zr) % n_planets:
                continue
            for delta in deltas:
                if (zr - zs - delta) % 2:
                    continue
                zp = (zr - zs - delta) // 2
                if zp < zp_min:
                    continue
                a = (zs + zp) / 2                       # centre distance per module
                if 2 * a * math.sin(math.pi / n_planets) < zp + 2 + neighbour_gap:
                    continue
                hunting = math.gcd(zs, zp) == 1 and math.gcd(zp, zr) == 1
                out.append(ToothCounts(zs, zp, zr, n_planets, i, err, hunting, delta))
    out.sort(key=lambda c: (round(abs(c.error) / 0.005), c.delta != 0, not c.hunting, c.zs))
    return out

### Example

A ratio of 5 with three planets. The list does not depend on the module, so it cannot tell which sun size is best; the design search in `gears.design` rates each candidate at each module. Here we take the first one with at least 17 sun teeth.

In [ ]:
cands = tooth_count_candidates(5.0, 3)
print(len(cands), "candidates")
for c in cands[:10]:
    print(f"zs={c.zs:2d} zp={c.zp:2d} zr={c.zr:3d}  i={c.ratio:.4f} ({c.error*100:+.2f} %)  "
          f"hunting={c.hunting}  δ={c.delta:+d}")
best = next(c for c in cands if c.zs >= 17)
print('chosen:', best)
assert (best.zs + best.zr) % 3 == 0 and best.zr == best.zs + 2 * best.zp + best.delta

## Profile Shifts: Theory

The sun is the smallest gear and carries the highest bending stress, so it gets a positive shift and the planet the opposite (a *zero-sum* pair keeps $a_w = a$):

$$x_s = \max\big(0.02\,(30 - z_s),\ x_{s,min} + 0.05\big)\ \text{clipped to }[0, 0.6],\qquad x_p = \max(-x_s,\ x_{p,min} + 0.05),$$

where $x_{min} = h_{fP}^* - \rho_{fP}^*(1-\sin\alpha) - \tfrac{z}{2}\sin^2\alpha$ is the undercut limit. The first term is a common rule for small pinions (it moves the start of active profile away from the weak root of a small gear) and is a starting point, not an optimum; you can always pass your own shifts.

The sun–planet mesh then fixes the centre distance $a_w$. The ring shift follows from requiring the planet–ring mesh to have the same $a_w$ (`shift_sum_for_centre_distance`). When the sun–planet shift sum is positive, both tips are shortened by the ISO 21771 amount $k^* = (x_s + x_p) - (a_w - a)/m$ to keep the standard tip clearance. Finally, if the ring tip would dig into the planet fillet, or the ring tip circle falls inside its base circle, the ring tip is shortened in steps of $0.05\,m$ until the mesh checks pass.

In [ ]:
#| export
def undercut_shift(z: int, hf: float = 1.25, rho_f: float = 0.38, alpha: float = math.radians(20.0)) -> float:
    "Smallest profile shift that avoids undercut of a rack-generated external gear with z teeth."
    return hf - rho_f * (1 - math.sin(alpha)) - z * math.sin(alpha) ** 2 / 2


def default_shifts(zs: int, zp: int) -> tuple[float, float]:
    "Starting profile shifts (x_sun, x_planet) for a planetary set; see the theory above."
    xs = min(max(0.02 * (30 - zs), undercut_shift(zs) + 0.05, 0.0), 0.6)
    xp = max(-xs, undercut_shift(zp) + 0.05)
    return xs, xp

In [ ]:
for zs in (12, 15, 17, 21, 30):
    print(zs, [round(v, 3) for v in default_shifts(zs, 25)], "undercut limit", round(undercut_shift(zs), 3))
assert default_shifts(12, 25)[0] > undercut_shift(12)

## `PlanetarySet`

In [ ]:
#| export
@dataclass(frozen=True)
class PlanetarySet:
    """
    A ring-fixed single-stage planetary gear set (sun in, carrier out).

    Attributes
    ----------
    sun, planet, ring : GearGeometry
    n_planets         : int
    sun_planet        : MeshCheck  Checks of the sun–planet mesh
    planet_ring       : MeshCheck  Checks of the planet–ring mesh
    """
    sun: GearGeometry
    planet: GearGeometry
    ring: GearGeometry
    n_planets: int
    sun_planet: MeshCheck
    planet_ring: MeshCheck

    @classmethod
    def build(cls, zs: int, zp: int, zr: int, m: float, n_planets: int, b: float,
              x_sun: float | None = None, x_planet: float | None = None,
              alpha: float = math.radians(20.0), b_ring: float | None = None) -> "PlanetarySet":
        """
        Build the set, choosing profile shifts and tip reductions as described above.
        `b` is the planet face width [mm]; the sun and ring default to the same width.
        """
        xs0, xp0 = default_shifts(zs, zp)
        xs = xs0 if x_sun is None else x_sun
        xp = xp0 if x_planet is None else x_planet
        sun = GearGeometry(z=zs, m=m, x=xs, b=b, alpha=alpha, name="sun")
        planet = GearGeometry(z=zp, m=m, x=xp, b=b, alpha=alpha, name="planet")
        a_w = centre_distance(sun, planet)
        k = (xs + xp) - (a_w - (zs + zp) * m / 2) / m           # ISO 21771 tip shortening
        if k > 1e-9:
            sun, planet = sun.with_(k_tip=k), planet.with_(k_tip=k)
        xr = shift_sum_for_centre_distance(zp, -zr, m, a_w, alpha) - xp
        ring = GearGeometry(z=zr, m=m, x=xr, b=b if b_ring is None else b_ring, alpha=alpha,
                            internal=True, name="ring")
        sp = mesh_check(sun, planet)
        pr = None
        for step in range(0, 21):
            kt = 0.05 * step
            trial = ring.with_(k_tip=kt)
            if trial.da < trial.db + 0.02 * m:
                continue
            pr = mesh_check(planet, trial)
            if pr.ok:
                ring = trial
                break
        if pr is None:
            raise ValueError("ring tip cannot be placed outside its base circle")
        if not pr.ok:
            ring = trial
        return cls(sun, planet, ring, n_planets, sp, pr)

    # --- kinematics -----------------------------------------------------------
    @property
    def ratio(self) -> float:
        "Speed ratio n_sun / n_carrier."
        return 1 + self.ring.z / self.sun.z

    @property
    def a_w(self) -> float:
        "Operating centre distance = planet pitch-circle radius on the carrier [mm]."
        return self.sun_planet.a_w

    def speeds(self, n_sun: float) -> dict[str, float]:
        "Speeds [rpm]: sun, carrier, planet about its own pin (relative to carrier), planet absolute."
        n_c = n_sun / self.ratio
        n_pc = -(n_sun - n_c) * self.sun.z / self.planet.z
        return {"sun": n_sun, "carrier": n_c, "planet_rel": n_pc, "planet_abs": n_pc + n_c}

    def mesh_frequency(self, n_sun: float) -> float:
        "Tooth-mesh frequency [Hz] (same for both meshes)."
        return abs(n_sun - n_sun / self.ratio) / 60 * self.sun.z

    # --- space checks ---------------------------------------------------------
    @property
    def assembles(self) -> bool:
        "Equal-spacing assembly condition (zs + zr)/N integer."
        return (self.sun.z + self.ring.z) % self.n_planets == 0

    @property
    def neighbour_gap(self) -> float:
        "Gap between the tip circles of neighbouring planets [mm] (room for carrier posts)."
        return 2 * self.a_w * math.sin(math.pi / self.n_planets) - self.planet.da

    @property
    def ok(self) -> bool:
        return (self.assembles and self.sun_planet.ok and self.planet_ring.ok
                and self.neighbour_gap > 0 and not self.sun.undercut and not self.planet.undercut
                and self.sun.sa > 0.25 * self.sun.m and self.planet.sa > 0.25 * self.planet.m)

    def problems(self) -> list[str]:
        "Readable list of the checks that fail."
        p = []
        if not self.assembles: p.append("planets cannot be equally spaced: (zs + zr)/N is not an integer")
        if self.neighbour_gap <= 0: p.append(f"neighbouring planets overlap by {-self.neighbour_gap:.2f} mm")
        for g in (self.sun, self.planet):
            if g.undercut: p.append(f"{g.name} is undercut (x = {g.x:+.3f})")
            if g.sa <= 0.25 * g.m: p.append(f"{g.name} tip is too thin (s_a = {g.sa:.3f} mm < 0.25 m)")
        for nm, mc in (("sun–planet", self.sun_planet), ("planet–ring", self.planet_ring)):
            if mc.eps_alpha < 1.2: p.append(f"{nm}: contact ratio {mc.eps_alpha:.2f} < 1.2")
            if mc.interference: p.append(f"{nm}: tip/root interference")
            if min(mc.clearance12, mc.clearance21) <= 0: p.append(f"{nm}: no tip clearance")
        return p

    def summary(self) -> str:
        "One-paragraph text summary of the set."
        g = self
        lines = [f"i = {g.ratio:.4f}, N = {g.n_planets}, m = {g.sun.m} mm, a_w = {g.a_w:.4f} mm",
                 f"  {'gear':7s} {'z':>4s} {'x':>7s} {'k*':>6s} {'d':>9s} {'da':>9s} {'df':>9s} {'b':>6s}"]
        for x in (g.sun, g.planet, g.ring):
            lines.append(f"  {x.name:7s} {x.z:4d} {x.x:+7.3f} {x.k_tip:6.3f} {x.d:9.3f} {x.da:9.3f} {x.df:9.3f} {x.b:6.1f}")
        lines.append(f"  ε_α sun–planet {g.sun_planet.eps_alpha:.3f}, planet–ring {g.planet_ring.eps_alpha:.3f}, "
                     f"α_w {math.degrees(g.sun_planet.alpha_w):.3f}° / {math.degrees(g.planet_ring.alpha_w):.3f}°, "
                     f"planet tip gap {g.neighbour_gap:.2f} mm")
        lines.append("  checks: " + ("all pass" if g.ok else "; ".join(g.problems())))
        return "\n".join(lines)

### Example

The best ratio-5 candidate above, module 1.5, three planets, 20 mm face width:

In [ ]:
ps = PlanetarySet.build(best.zs, best.zp, best.zr, m=1.5, n_planets=3, b=20)
print(ps.summary())
print({k: round(v, 1) for k, v in ps.speeds(3000).items()}, f"mesh frequency {ps.mesh_frequency(3000):.0f} Hz")
assert ps.ok and math.isclose(ps.planet_ring.a_w, ps.a_w, rel_tol=1e-12)

A set with δ = ±1 (ring one tooth off the unshifted coaxial count) still builds on one centre distance, through the shifts:

In [ ]:
odd = next(c for c in tooth_count_candidates(5.6, 3, tol=0.03) if c.delta != 0)
ps_odd = PlanetarySet.build(odd.zs, odd.zp, odd.zr, m=1.0, n_planets=3, b=12)
print(odd)
print(ps_odd.summary())
assert math.isclose(ps_odd.planet_ring.a_w, ps_odd.a_w, rel_tol=1e-12)

## Module Limits From the Envelope and the Sun Shaft: Theory

**Largest module (envelope).** If the ring gear is also the outer wall, its outside diameter is the root diameter plus two rim thicknesses $s_R$:

$$D_{ring} \approx m\,(z_r + 2h_{fP}^*) + 2 s_R \le D_{env} - 2t_{housing}.$$

ISO 6336-3 applies no rim penalty to an internal gear when $s_R \ge 3.5\,m$, so with $s_R = c_R\,m$

$$m_{max} = \frac{D_{env} - 2t_{housing}}{z_r + 2h_{fP}^* + 2c_R}.$$

**Smallest module (sun bore).** The sun sits on the input shaft of diameter $d_{sh}$ with a key whose hub groove is $t_2$ deep (DIN 6885-1). The rim between the sun root and the bottom of the key groove must be at least $c_s$ tooth depths $h_t = (h_{aP}^* + h_{fP}^*)m$:

$$\frac{d_{f,s}}{2} - \frac{d_{sh}}{2} - t_2 \ge c_s h_t,\qquad m_{min} = \frac{d_{sh}/2 + t_2}{z_s/2 - h_{fP}^* - c_s (h_{aP}^* + h_{fP}^*)}.$$

$c_s = 1.2$ gives no rim penalty for an external gear; values down to 0.5 are allowed by ISO 6336-3 with a rim factor $Y_B > 1$ on the bending stress (`gears.rating` applies it). A positive sun shift raises $d_{f,s}$ and helps here; the formula ignores it and so errs on the safe side. When no module satisfies both limits, the sun has to be cut on the shaft itself (pinion shaft) or the shaft–hub joint changed (spline instead of key).

In [ ]:
#| export
def module_limits(envelope_d: float, shaft_d: float, zs: int, zr: int, t2: float = 0.0,
                  housing_wall: float = 0.0, ring_rim: float = 3.5, sun_rim: float = 1.2,
                  ha: float = 1.0, hf: float = 1.25) -> tuple[float, float]:
    """
    (m_min, m_max) [mm] from the sun bore and the envelope, see the theory above.

    envelope_d   : outside diameter available for the ring (or ring + housing) [mm]
    shaft_d      : shaft diameter in the sun bore [mm]
    t2           : hub key-groove depth [mm] (0 for a plain or splined bore)
    housing_wall : housing wall thickness outside the ring [mm] (0 if the ring is the housing)
    ring_rim     : ring rim thickness under the tooth roots, in modules
    sun_rim      : sun rim thickness above the key groove, in tooth depths
    """
    m_max = (envelope_d - 2 * housing_wall) / (zr + 2 * hf + 2 * ring_rim)
    den = zs / 2 - hf - sun_rim * (ha + hf)
    m_min = math.inf if den <= 0 else (shaft_d / 2 + t2) / den
    return m_min, m_max

### Example

A 120 mm envelope and a 14 mm keyed input shaft (DIN 6885-1 key 5 × 5, hub groove $t_2 = 2.3$ mm). For a ratio of 5 the ring has about four times the sun's teeth, so a small sun leaves no room for the key and a large one pushes the ring past the envelope. Only sets with enough sun teeth leave a window, and the window has to contain a standard module:

In [ ]:
from emachines.gears.involute import standard_modules
feasible, shown = [], 0
for c in cands:
    m_lo, m_hi = module_limits(envelope_d=120, shaft_d=14, zs=c.zs, zr=c.zr, t2=2.3)
    ms = standard_modules(m_lo, m_hi, series_ii=True)
    if shown < 12 and (c.zs in (15, 18, 21, 24) or ms):
        shown += 1
        print(f"zs={c.zs:2d} zp={c.zp:2d} zr={c.zr:3d}: m from {m_lo:.3f} to {m_hi:.3f} mm  standard: {ms}")
    if ms:
        feasible.append((c, ms))
print(f'{len(feasible)} of {len(cands)} candidates fit')
assert feasible and all(module_limits(120, 14, c.zs, c.zr, 2.3)[0] <= ms[0] for c, ms in feasible)

---

## Tests

In [ ]:
#| hide
# every candidate satisfies the conditions it claims
for c in tooth_count_candidates(4.2, 4, tol=0.03) + tooth_count_candidates(7.1, 3, tol=0.03):
    assert (c.zs + c.zr) % c.n_planets == 0 and c.zr - c.zs - 2 * c.zp == c.delta
    assert abs(1 + c.zr / c.zs - 4.2) / 4.2 <= 0.03 or abs(1 + c.zr / c.zs - 7.1) / 7.1 <= 0.03
# classic 3-planet set 18/27/72 is found for i = 5 (18+72 = 90, divisible by 3)
assert any((c.zs, c.zp, c.zr) == (18, 27, 72) for c in tooth_count_candidates(5.0, 3, tol=0.001))
# Willis: planet absolute speed equation with ring fixed; ring speed 0 recovered
s = ps.speeds(1000.0)
n_r = s["carrier"] - (s["sun"] - s["carrier"]) * ps.sun.z / ps.ring.z
assert abs(n_r) < 1e-9
# planet relative speed times its teeth = sun relative speed times its teeth (same mesh)
assert math.isclose(abs(s["planet_rel"]) * ps.planet.z, (s["sun"] - s["carrier"]) * ps.sun.z)
# module limits respond in the right direction
assert module_limits(150, 14, 18, 72)[1] > module_limits(120, 14, 18, 72)[1]
assert module_limits(120, 20, 18, 72)[0] > module_limits(120, 14, 18, 72)[0]
# zero-sum default shifts on a delta = 0 set keep a_w = a
p0 = PlanetarySet.build(18, 27, 72, 1.0, 3, 10)
assert math.isclose(p0.a_w, 22.5) and p0.ok
# the problems list is empty exactly when ok
assert (p0.problems() == []) == p0.ok

## References

- Müller, H.W. (1982). *Epicyclic Drive Trains*. Wayne State University Press — Willis equation, assembly and neighbour conditions.
- ISO 21771:2007 — tip shortening for gear pairs with a positive shift sum.
- ISO 6336-3:2019, rim thickness factor $Y_B$ — limits on the rim thickness of the sun and ring.
- DIN 3992:1964 — choice of profile shift coefficients for external gears.